# 04B - Modelado V2 en cluster

Notebook pensado para ejecutar la busqueda masiva de modelos en un cluster con aproximadamente:

- 60 hilos CPU
- 100 GB RAM
- opcionalmente GPU para XGBoost

Objetivo: predecir `susceptibility` binaria:

- `Resistant`
- `Susceptible`

Este notebook esta separado del `04_modelado_multibacteria_armd.ipynb` local para no romper la version de trabajo en tu PC.

## Estrategia de cluster

No se usa `n_jobs=-1` en todo porque eso puede causar sobreparalelizacion: por ejemplo, GridSearchCV abre muchos procesos y cada modelo interno intenta usar todos los hilos otra vez.

Configuracion recomendada para tu cluster:

- `N_JOBS_GRID = 40`: paraleliza configuraciones/folds sin ocupar los 60 hilos completos.
- `N_JOBS_MODELO = 1`: los modelos internos no abren mas hilos cuando GridSearchCV ya esta paralelizando.
- `N_JOBS_XGBOOST = 12`: XGBoost puede usar mas hilos internos, pero sin saturar todo.
- `CONFIGURACIONES_POR_MODELO = 200`: busqueda amplia por familia de modelo.

El notebook guarda resultados por modelo. Si un modelo falla o se corta el job, puedes volver a ejecutar y saltara los modelos ya terminados cuando `REANUDAR = True`.


In [ ]:
# Esta celda debe ejecutarse antes de importar numpy/sklearn para controlar hilos nativos.
import os

N_HILOS_CLUSTER = 60
N_JOBS_GRID = 40
N_JOBS_MODELO = 1
N_JOBS_XGBOOST = 12
CONFIGURACIONES_POR_MODELO = 200
REANUDAR = True
USAR_CUDA_XGBOOST = True
DATASET_USAR = "balanceado_organismo_clase"
RANDOM_STATE = 42

# Evita que BLAS/OpenMP use 60 hilos dentro de cada proceso de GridSearchCV.
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

print("Configuracion cluster")
print("N_HILOS_CLUSTER:", N_HILOS_CLUSTER)
print("N_JOBS_GRID:", N_JOBS_GRID)
print("N_JOBS_MODELO:", N_JOBS_MODELO)
print("N_JOBS_XGBOOST:", N_JOBS_XGBOOST)
print("CONFIGURACIONES_POR_MODELO:", CONFIGURACIONES_POR_MODELO)
print("REANUDAR:", REANUDAR)


In [ ]:
from pathlib import Path
import json
import warnings
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, precision_score, recall_score,
    mean_squared_error, mean_absolute_error, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_auc_score, average_precision_score, log_loss,
    brier_score_loss, matthews_corrcoef, cohen_kappa_score, fbeta_score,
    roc_curve, precision_recall_curve, auc,
)
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import ComplementNB, GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier,
    AdaBoostClassifier, GradientBoostingClassifier,
)
from sklearn.neural_network import MLPClassifier
from sklearn.inspection import permutation_importance
from sklearn.utils.class_weight import compute_sample_weight

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")


## Como correr esto en el cluster

### Opcion A: JupyterLab

1. Copia o sube el proyecto completo al cluster.
2. Asegurate de que exista `modelo/V2/DATOS_PROCESADOS/09_dataset_v2_multibacteria_balanceado_organismo_clase.csv`.
3. Abre este notebook desde JupyterLab.
4. Ejecuta todas las celdas.
5. Si se corta el job, vuelve a ejecutar. Con `REANUDAR = True`, saltara modelos que ya tienen CSV de resultado.

### Opcion B: terminal con nbconvert

Desde la raiz del proyecto:

```bash
jupyter nbconvert --to notebook --execute modelo/V2/04_MODELADO/04B_modelado_cluster_multibacteria_armd.ipynb --output 04B_modelado_cluster_multibacteria_armd_ejecutado.ipynb --ExecutePreprocessor.timeout=-1
```

### Opcion C: SLURM ejemplo

```bash
#!/bin/bash
#SBATCH --job-name=armd-v2-modelos
#SBATCH --cpus-per-task=60
#SBATCH --mem=100G
#SBATCH --time=24:00:00
#SBATCH --output=logs/armd_v2_%j.out
#SBATCH --error=logs/armd_v2_%j.err

source ~/.bashrc
conda activate armd-ai
cd /ruta/al/Proyecto
jupyter nbconvert --to notebook --execute modelo/V2/04_MODELADO/04B_modelado_cluster_multibacteria_armd.ipynb --output 04B_modelado_cluster_multibacteria_armd_ejecutado.ipynb --ExecutePreprocessor.timeout=-1
```


In [ ]:
RUTA_ACTUAL = Path.cwd().resolve()
RUTA_MODELO = None
for candidata in [RUTA_ACTUAL, *RUTA_ACTUAL.parents]:
    if (candidata / "V2" / "DATOS_PROCESADOS").exists() and candidata.name.lower() == "modelo":
        RUTA_MODELO = candidata
        break
    if (candidata / "modelo" / "V2" / "DATOS_PROCESADOS").exists():
        RUTA_MODELO = candidata / "modelo"
        break
if RUTA_MODELO is None:
    raise FileNotFoundError("No se encontro modelo/V2/DATOS_PROCESADOS. Ejecuta primero notebooks 01-03 de V2.")

RUTA_V2 = RUTA_MODELO / "V2"
RUTA_PROCESADOS = RUTA_V2 / "DATOS_PROCESADOS"
RUTA_RESULTADOS = RUTA_V2 / "05_RESULTADOS"
RUTA_RESULTADOS_CLUSTER = RUTA_RESULTADOS / "cluster"
RUTA_GRAFICAS = RUTA_V2 / "GRAFICAS"
RUTA_GRAFICAS_CLUSTER = RUTA_GRAFICAS / "cluster"

RUTA_RESULTADOS_CLUSTER.mkdir(parents=True, exist_ok=True)
RUTA_GRAFICAS_CLUSTER.mkdir(parents=True, exist_ok=True)

rutas_dataset = {
    "completo": RUTA_PROCESADOS / "07_dataset_v2_multibacteria_completo.csv",
    "balanceado_clase": RUTA_PROCESADOS / "08_dataset_v2_multibacteria_balanceado_clase.csv",
    "balanceado_organismo_clase": RUTA_PROCESADOS / "09_dataset_v2_multibacteria_balanceado_organismo_clase.csv",
}
RUTA_DATASET = rutas_dataset[DATASET_USAR]
print("Dataset:", RUTA_DATASET)
print("Resultados cluster:", RUTA_RESULTADOS_CLUSTER)
print("Graficas cluster:", RUTA_GRAFICAS_CLUSTER)


In [ ]:
df = pd.read_csv(RUTA_DATASET, low_memory=False)

ruta_decisiones = RUTA_PROCESADOS / "17_decision_variables_v2.csv"
if ruta_decisiones.exists():
    decisiones = pd.read_csv(ruta_decisiones)
    variables_excluidas = set(decisiones.loc[decisiones["decision"].eq("excluir"), "variable"])
else:
    variables_excluidas = set()

columnas_exp_prev = [c for c in df.columns if c.startswith("exp_prev_")]
if not columnas_exp_prev:
    raise ValueError("El dataset V2 no tiene columnas exp_prev_*. Ejecuta primero el notebook 02.")

columnas_excluir = {"anon_id", "pat_enc_csn_id_coded", "order_proc_id_coded", "order_time_jittered_utc", "susceptibility"}
columnas_excluir = columnas_excluir.union(variables_excluidas)

X = df.drop(columns=[c for c in columnas_excluir if c in df.columns])
y_texto = df["susceptibility"].astype(str)

le = LabelEncoder()
y = le.fit_transform(y_texto)

columnas_categoricas = X.select_dtypes(include=["object", "string"]).columns.tolist()
columnas_numericas = [c for c in X.columns if c not in columnas_categoricas]

print("Filas:", len(df))
print("Predictores:", X.shape[1])
print("Categoricas:", len(columnas_categoricas))
print("Numericas:", len(columnas_numericas))
print("Columnas exp_prev_*:", len(columnas_exp_prev))
print("Clases:", dict(zip(le.classes_, le.transform(le.classes_))))
display(y_texto.value_counts())


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

preprocesador_escalado = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), columnas_numericas),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="constant", fill_value="SIN_REGISTRO")), ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=20))]), columnas_categoricas),
])

preprocesador_minmax = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", MinMaxScaler())]), columnas_numericas),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="constant", fill_value="SIN_REGISTRO")), ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=20))]), columnas_categoricas),
])

preprocesador_arboles = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), columnas_numericas),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="constant", fill_value="SIN_REGISTRO")), ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=20))]), columnas_categoricas),
])

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
pesos_train = compute_sample_weight(class_weight="balanced", y=y_train)

print("Train:", X_train.shape, "Test:", X_test.shape)


In [ ]:
def seleccionar_configuraciones_uniformes(configuraciones, n=CONFIGURACIONES_POR_MODELO):
    """Selecciona configuraciones repartidas por toda la grilla, no solo las primeras."""
    if len(configuraciones) < n:
        raise ValueError(f"Cada modelo debe tener al menos {n} configuraciones, recibio {len(configuraciones)}")
    if len(configuraciones) == n:
        return configuraciones
    indices = np.linspace(0, len(configuraciones) - 1, n, dtype=int)
    return [configuraciones[i] for i in indices]


def construir_grid(configuraciones, esperado=CONFIGURACIONES_POR_MODELO):
    configuraciones = seleccionar_configuraciones_uniformes(configuraciones, esperado)
    return [{k: [v] for k, v in config.items()} for config in configuraciones]


In [ ]:
# Grillas amplias. Cada familia queda reducida a CONFIGURACIONES_POR_MODELO puntos uniformemente distribuidos.
config_lr = []
for C in np.geomspace(0.003, 100.0, 50):
    config_lr.append({"clf__C": C, "clf__fit_intercept": True, "clf__solver": "lbfgs", "clf__penalty": "l2", "clf__tol": 1e-4})
    config_lr.append({"clf__C": C, "clf__fit_intercept": False, "clf__solver": "lbfgs", "clf__penalty": "l2", "clf__tol": 1e-4})
    for l1_ratio in [0.15, 0.50, 0.85]:
        config_lr.append({"clf__C": C, "clf__fit_intercept": True, "clf__solver": "saga", "clf__penalty": "elasticnet", "clf__l1_ratio": l1_ratio, "clf__tol": 1e-3})

config_svm_linear = [
    {"clf__C": C, "clf__tol": tol, "clf__loss": loss, "clf__class_weight": class_weight, "clf__max_iter": 6000}
    for C in np.geomspace(0.005, 30.0, 25)
    for tol in [1e-4, 5e-4]
    for loss in ["hinge", "squared_hinge"]
    for class_weight in ["balanced", None]
]

config_sgd = [
    {"clf__loss": loss, "clf__alpha": alpha, "clf__penalty": penalty, "clf__l1_ratio": l1_ratio, "clf__class_weight": "balanced", "clf__max_iter": 2500, "clf__tol": 1e-3}
    for loss in ["log_loss", "modified_huber", "hinge"]
    for alpha in np.geomspace(1e-6, 1e-2, 20)
    for penalty in ["l2", "elasticnet"]
    for l1_ratio in [0.15, 0.50]
]

config_knn = [
    {"svd__n_components": n_comp, "clf__n_neighbors": k, "clf__weights": weights, "clf__p": p}
    for n_comp in [30, 50, 80, 120, 160]
    for k in [3, 5, 7, 11, 15, 21, 31, 45, 61, 81]
    for weights in ["uniform", "distance"]
    for p in [1, 2]
]

config_cnb = [
    {"clf__alpha": alpha, "clf__fit_prior": fit_prior, "clf__norm": norm}
    for alpha in np.geomspace(1e-4, 100.0, 50)
    for fit_prior in [True, False]
    for norm in [True, False]
]

config_gnb = [
    {"svd__n_components": n_comp, "clf__var_smoothing": var_smoothing}
    for n_comp in [30, 50, 80, 120]
    for var_smoothing in np.geomspace(1e-12, 1e-2, 50)
]

config_dt = [
    {"clf__max_depth": max_depth, "clf__min_samples_leaf": min_leaf, "clf__criterion": criterion, "clf__ccp_alpha": ccp_alpha, "clf__min_samples_split": min_split}
    for max_depth in [4, 6, 8, 10, 12, 16, 20, 28, None]
    for min_leaf in [5, 10, 20, 40, 80, 120]
    for min_split in [2, 10]
    for criterion in ["gini", "entropy", "log_loss"]
    for ccp_alpha in [0.0, 0.0002, 0.001]
]

config_rf = [
    {"clf__n_estimators": n_estimators, "clf__max_depth": max_depth, "clf__min_samples_leaf": min_leaf, "clf__max_features": max_features, "clf__bootstrap": bootstrap}
    for n_estimators in [160, 240, 320, 420, 560]
    for max_depth in [12, 18, 24, None]
    for min_leaf in [2, 5, 10, 20, 40]
    for max_features in ["sqrt", "log2"]
    for bootstrap in [True, False]
]

config_extra_trees = [
    {"clf__n_estimators": n_estimators, "clf__max_depth": max_depth, "clf__min_samples_leaf": min_leaf, "clf__max_features": max_features, "clf__bootstrap": bootstrap}
    for n_estimators in [160, 240, 320, 420, 560]
    for max_depth in [12, 18, 24, None]
    for min_leaf in [2, 5, 10, 20, 40]
    for max_features in ["sqrt", "log2"]
    for bootstrap in [False, True]
]

config_adaboost = [
    {"clf__n_estimators": n_estimators, "clf__learning_rate": learning_rate, "clf__estimator__max_depth": depth, "clf__estimator__min_samples_leaf": min_leaf}
    for n_estimators in [80, 120, 180, 240, 320]
    for learning_rate in [0.01, 0.03, 0.05, 0.08, 0.12]
    for depth in [1, 2, 3, 4]
    for min_leaf in [10, 30, 60, 100]
]

config_gb = [
    {"clf__n_estimators": n_estimators, "clf__learning_rate": learning_rate, "clf__max_depth": depth, "clf__min_samples_leaf": min_leaf, "clf__subsample": subsample}
    for n_estimators in [80, 120, 180, 240]
    for learning_rate in [0.02, 0.04, 0.06, 0.10, 0.15]
    for depth in [2, 3, 4, 5]
    for min_leaf in [10, 30, 60]
    for subsample in [0.75, 0.90, 1.0]
]

config_hgb = [
    {"clf__max_iter": max_iter, "clf__learning_rate": learning_rate, "clf__max_leaf_nodes": max_leaf_nodes, "clf__l2_regularization": l2, "clf__min_samples_leaf": min_leaf}
    for max_iter in [120, 180, 240, 320, 440]
    for learning_rate in [0.02, 0.03, 0.05, 0.08, 0.12]
    for max_leaf_nodes in [15, 31, 63, 95]
    for l2 in [0.0, 0.1, 1.0, 2.0]
    for min_leaf in [20, 40]
]

config_mlp = [
    {"svd__n_components": n_comp, "clf__hidden_layer_sizes": hidden, "clf__alpha": alpha, "clf__learning_rate_init": lr, "clf__activation": activation}
    for n_comp in [50, 80, 120, 160]
    for hidden in [(32,), (64,), (96,), (64, 32), (128, 64)]
    for alpha in [1e-5, 1e-4, 1e-3, 1e-2]
    for lr in [0.0005, 0.001, 0.003]
    for activation in ["relu", "tanh"]
]

config_xgb = [
    {
        "clf__n_estimators": n_estimators,
        "clf__max_depth": max_depth,
        "clf__learning_rate": learning_rate,
        "clf__subsample": subsample,
        "clf__colsample_bytree": colsample,
        "clf__min_child_weight": min_child_weight,
        "clf__reg_lambda": reg_lambda,
        "clf__reg_alpha": reg_alpha,
    }
    for n_estimators in [160, 240, 320, 440, 600]
    for max_depth in [3, 4, 5, 6, 8]
    for learning_rate in [0.02, 0.03, 0.05, 0.08, 0.12]
    for subsample in [0.75, 0.85, 0.95]
    for colsample in [0.75, 0.85, 0.95]
    for min_child_weight in [1, 3, 5]
    for reg_lambda in [1.0, 2.0, 5.0]
    for reg_alpha in [0.0, 0.1]
]


In [ ]:
modelos_parametros = {
    "logistic_regression": (
        Pipeline([("prep", preprocesador_escalado), ("clf", LogisticRegression(max_iter=2500, class_weight="balanced", n_jobs=N_JOBS_MODELO, random_state=RANDOM_STATE))]),
        construir_grid(config_lr),
    ),
    "svm_linear": (
        Pipeline([("prep", preprocesador_escalado), ("clf", LinearSVC(random_state=RANDOM_STATE, dual="auto"))]),
        construir_grid(config_svm_linear),
    ),
    "sgd_linear": (
        Pipeline([("prep", preprocesador_escalado), ("clf", SGDClassifier(random_state=RANDOM_STATE, n_jobs=N_JOBS_MODELO))]),
        construir_grid(config_sgd),
    ),
    "knn_svd": (
        Pipeline([("prep", preprocesador_escalado), ("svd", TruncatedSVD(random_state=RANDOM_STATE)), ("clf", KNeighborsClassifier(algorithm="brute", n_jobs=N_JOBS_MODELO))]),
        construir_grid(config_knn),
    ),
    "complement_naive_bayes": (
        Pipeline([("prep", preprocesador_minmax), ("clf", ComplementNB())]),
        construir_grid(config_cnb),
    ),
    "gaussian_naive_bayes_svd": (
        Pipeline([("prep", preprocesador_escalado), ("svd", TruncatedSVD(random_state=RANDOM_STATE)), ("clf", GaussianNB())]),
        construir_grid(config_gnb),
    ),
    "decision_tree": (
        Pipeline([("prep", preprocesador_arboles), ("clf", DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE))]),
        construir_grid(config_dt),
    ),
    "random_forest": (
        Pipeline([("prep", preprocesador_arboles), ("clf", RandomForestClassifier(class_weight="balanced_subsample", random_state=RANDOM_STATE, n_jobs=N_JOBS_MODELO))]),
        construir_grid(config_rf),
    ),
    "extra_trees": (
        Pipeline([("prep", preprocesador_arboles), ("clf", ExtraTreesClassifier(class_weight="balanced", random_state=RANDOM_STATE, n_jobs=N_JOBS_MODELO))]),
        construir_grid(config_extra_trees),
    ),
    "adaboost_tree": (
        Pipeline([("prep", preprocesador_arboles), ("clf", AdaBoostClassifier(estimator=DecisionTreeClassifier(random_state=RANDOM_STATE), random_state=RANDOM_STATE))]),
        construir_grid(config_adaboost),
    ),
    "gradient_boosting": (
        Pipeline([("prep", preprocesador_arboles), ("clf", GradientBoostingClassifier(random_state=RANDOM_STATE))]),
        construir_grid(config_gb),
    ),
    "hist_gradient_boosting": (
        Pipeline([("prep", preprocesador_arboles), ("clf", HistGradientBoostingClassifier(random_state=RANDOM_STATE, class_weight="balanced"))]),
        construir_grid(config_hgb),
    ),
    "mlp_svd": (
        Pipeline([("prep", preprocesador_escalado), ("svd", TruncatedSVD(random_state=RANDOM_STATE)), ("clf", MLPClassifier(max_iter=120, early_stopping=True, random_state=RANDOM_STATE))]),
        construir_grid(config_mlp),
    ),
    "xgboost": (
        Pipeline([("prep", preprocesador_arboles), ("clf", XGBClassifier(
            objective="binary:logistic", eval_metric="logloss", tree_method="hist",
            device="cuda" if USAR_CUDA_XGBOOST else "cpu", random_state=RANDOM_STATE, n_jobs=N_JOBS_XGBOOST,
        ))]),
        construir_grid(config_xgb),
    ),
}

resumen_configuraciones = pd.DataFrame([
    {"modelo": nombre, "configuraciones": len(param_grid), "fits_cv_estimados": len(param_grid) * cv.get_n_splits()}
    for nombre, (_, param_grid) in modelos_parametros.items()
])
resumen_configuraciones.to_csv(RUTA_RESULTADOS_CLUSTER / f"00_configuraciones_cluster_v2_binario_{DATASET_USAR}.csv", index=False)
display(resumen_configuraciones)
print("Total configuraciones:", int(resumen_configuraciones["configuraciones"].sum()))
print("Total fits CV estimados:", int(resumen_configuraciones["fits_cv_estimados"].sum()))


## Seleccion de modelos a correr

Por defecto corre todos. Si el cluster tiene ventana corta, puedes poner por ejemplo:

```python
MODELOS_A_EJECUTAR = ["random_forest", "extra_trees", "hist_gradient_boosting", "xgboost"]
```

Para la busqueda masiva completa, deja `MODELOS_A_EJECUTAR = list(modelos_parametros.keys())`.


In [ ]:
MODELOS_A_EJECUTAR = list(modelos_parametros.keys())
MODELOS_A_EJECUTAR


In [ ]:

def obtener_proba_resistant(modelo, X_eval, indice_resistant):
    """Devuelve probabilidad de Resistant si el modelo la soporta."""
    if not hasattr(modelo, "predict_proba"):
        return None
    try:
        proba = modelo.predict_proba(X_eval)
        clases_modelo = modelo.named_steps["clf"].classes_ if hasattr(modelo, "named_steps") else modelo.classes_
        pos = int(np.where(clases_modelo == indice_resistant)[0][0])
        return proba[:, pos]
    except Exception:
        return None


def entrenar_modelo_cluster(nombre, pipeline, params):
    ruta_cv = RUTA_RESULTADOS_CLUSTER / f"cv_detalle_{nombre}_cluster_v2_binario_{DATASET_USAR}.csv"
    ruta_resumen = RUTA_RESULTADOS_CLUSTER / f"resultado_{nombre}_cluster_v2_binario_{DATASET_USAR}.csv"
    ruta_reporte = RUTA_RESULTADOS_CLUSTER / f"reporte_{nombre}_cluster_v2_binario_{DATASET_USAR}.json"

    if REANUDAR and ruta_cv.exists() and ruta_resumen.exists() and ruta_reporte.exists():
        print(f"Saltando {nombre}: resultados existentes")
        return pd.read_csv(ruta_resumen), pd.read_csv(ruta_cv)

    print(f"\n=== Entrenando {nombre} ===")
    inicio = time.time()
    busqueda = GridSearchCV(
        estimator=pipeline,
        param_grid=params,
        scoring="f1_macro",
        cv=cv,
        n_jobs=N_JOBS_GRID if nombre != "xgboost" else 1,
        verbose=2,
        error_score="raise",
        return_train_score=True,
    )

    try:
        if nombre == "xgboost":
            busqueda.fit(X_train, y_train, clf__sample_weight=pesos_train)
        else:
            busqueda.fit(X_train, y_train)
    except Exception as error:
        if nombre == "xgboost" and USAR_CUDA_XGBOOST:
            print("XGBoost CUDA fallo, reintentando en CPU:", error)
            pipeline.set_params(clf__device="cpu")
            busqueda = GridSearchCV(
                estimator=pipeline,
                param_grid=params,
                scoring="f1_macro",
                cv=cv,
                n_jobs=1,
                verbose=2,
                error_score="raise",
                return_train_score=True,
            )
            busqueda.fit(X_train, y_train, clf__sample_weight=pesos_train)
        else:
            raise

    tiempo_min = (time.time() - inicio) / 60
    mejor = busqueda.best_estimator_
    pred = mejor.predict(X_test)

    indice_resistant = int(np.where(le.classes_ == "Resistant")[0][0])
    y_bin = (np.asarray(y_test) == indice_resistant).astype(int)
    pred_bin = (np.asarray(pred) == indice_resistant).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_bin, pred_bin, labels=[0, 1]).ravel()
    sensibilidad_resistant = tp / (tp + fn) if (tp + fn) else np.nan
    especificidad_resistant = tn / (tn + fp) if (tn + fp) else np.nan
    ppv_resistant = tp / (tp + fp) if (tp + fp) else np.nan
    npv_resistant = tn / (tn + fn) if (tn + fn) else np.nan

    proba_resistant = obtener_proba_resistant(mejor, X_test, indice_resistant)
    if proba_resistant is not None:
        eps = 1e-15
        proba_clip = np.clip(proba_resistant, eps, 1 - eps)
        roc_auc = roc_auc_score(y_bin, proba_resistant)
        pr_auc = average_precision_score(y_bin, proba_resistant)
        logloss = log_loss(y_bin, proba_clip, labels=[0, 1])
        brier = brier_score_loss(y_bin, proba_resistant)
    else:
        roc_auc = np.nan
        pr_auc = np.nan
        logloss = np.nan
        brier = np.nan

    fila = {
        "dataset": DATASET_USAR,
        "modelo": nombre,
        "configuraciones_probadas": len(params),
        "fits_cv_estimados": len(params) * cv.get_n_splits(),
        "tiempo_minutos": tiempo_min,
        "cv_f1_macro": busqueda.best_score_,
        "test_accuracy": accuracy_score(y_test, pred),
        "test_balanced_accuracy": balanced_accuracy_score(y_test, pred),
        "test_f1_macro": f1_score(y_test, pred, average="macro", zero_division=0),
        "test_f1_weighted": f1_score(y_test, pred, average="weighted", zero_division=0),
        "test_precision_macro": precision_score(y_test, pred, average="macro", zero_division=0),
        "test_recall_macro": recall_score(y_test, pred, average="macro", zero_division=0),
        "test_sensibilidad_resistant": sensibilidad_resistant,
        "test_especificidad_resistant": especificidad_resistant,
        "test_ppv_resistant": ppv_resistant,
        "test_npv_resistant": npv_resistant,
        "test_mcc": matthews_corrcoef(y_test, pred),
        "test_cohen_kappa": cohen_kappa_score(y_test, pred),
        "test_mse": mean_squared_error(y_test, pred),
        "test_mae": mean_absolute_error(y_test, pred),
        "test_roc_auc_resistant": roc_auc,
        "test_pr_auc_resistant": pr_auc,
        "test_log_loss_resistant": logloss,
        "test_brier_score_resistant": brier,
        "tn_susceptible_correcto": tn,
        "fp_resistant_falso": fp,
        "fn_resistant_no_detectado": fn,
        "tp_resistant_correcto": tp,
        "mejores_parametros": json.dumps(busqueda.best_params_, ensure_ascii=False),
    }

    df_resumen = pd.DataFrame([fila])
    df_cv = pd.DataFrame(busqueda.cv_results_)
    df_cv["modelo"] = nombre
    df_cv["dataset"] = DATASET_USAR
    df_cv["gap_train_test"] = df_cv["mean_train_score"] - df_cv["mean_test_score"]

    reporte = classification_report(y_test, pred, target_names=le.classes_, output_dict=True, zero_division=0)

    df_cv.to_csv(ruta_cv, index=False)
    df_resumen.to_csv(ruta_resumen, index=False)
    with ruta_reporte.open("w", encoding="utf-8") as f:
        json.dump(reporte, f, ensure_ascii=False, indent=2)

    print(f"Terminado {nombre} en {tiempo_min:.1f} min")
    display(df_resumen)
    return df_resumen, df_cv


In [ ]:
resultados = []
cv_detallado = []

for nombre in MODELOS_A_EJECUTAR:
    pipeline, params = modelos_parametros[nombre]
    df_resumen, df_cv = entrenar_modelo_cluster(nombre, pipeline, params)
    resultados.append(df_resumen)
    cv_detallado.append(df_cv)

resultados_df = pd.concat(resultados, ignore_index=True).sort_values("test_f1_macro", ascending=False)
cv_detallado_df = pd.concat(cv_detallado, ignore_index=True)

resultados_df.to_csv(RUTA_RESULTADOS_CLUSTER / f"01_resultados_modelos_cluster_v2_binario_{DATASET_USAR}.csv", index=False)
cv_detallado_df.to_csv(RUTA_RESULTADOS_CLUSTER / f"05_cv_detalle_todos_modelos_cluster_v2_binario_{DATASET_USAR}.csv", index=False)

display(resultados_df)


In [ ]:
indice_resistant = int(np.where(le.classes_ == "Resistant")[0][0])
indice_susceptible = int(np.where(le.classes_ == "Susceptible")[0][0])

# Para no reentrenar, esta seccion calcula diagnostico desde los CSV de CV ya generados.
mejores_cv_por_modelo = (
    cv_detallado_df.sort_values(["modelo", "mean_test_score"], ascending=[True, False])
    .groupby("modelo", as_index=False)
    .head(1)
    [["modelo", "mean_test_score", "std_test_score", "mean_train_score", "gap_train_test", "params"]]
    .sort_values("mean_test_score", ascending=False)
)
mejores_cv_por_modelo["riesgo_overfitting"] = pd.cut(
    mejores_cv_por_modelo["gap_train_test"],
    bins=[-np.inf, 0.03, 0.07, np.inf],
    labels=["bajo", "moderado", "alto"],
)
mejores_cv_por_modelo.to_csv(RUTA_RESULTADOS_CLUSTER / f"08_diagnostico_overfitting_cv_cluster_v2_binario_{DATASET_USAR}.csv", index=False)
display(mejores_cv_por_modelo)


In [ ]:
# Graficas resumen de cluster.
fig, ax = plt.subplots(figsize=(12, 7))
plot_df = resultados_df.sort_values("test_f1_macro", ascending=True)
ax.barh(plot_df["modelo"], plot_df["test_f1_macro"], color="#315f72")
ax.set_xlabel("F1 macro en test")
ax.set_title("Ranking de modelos V2 - ejecucion cluster")
for i, valor in enumerate(plot_df["test_f1_macro"]):
    ax.text(valor + 0.002, i, f"{valor:.3f}", va="center")
fig.tight_layout()
fig.savefig(RUTA_GRAFICAS_CLUSTER / f"01_cluster_ranking_modelos_v2_binario_{DATASET_USAR}.png", dpi=220, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(12, 7))
gap_plot = mejores_cv_por_modelo.sort_values("gap_train_test", ascending=True)
ax.barh(gap_plot["modelo"], gap_plot["gap_train_test"], color="#9a6b2f")
ax.axvline(0.03, color="#3f7d5c", linestyle="--", linewidth=1, label="bajo/moderado")
ax.axvline(0.07, color="#a33f3f", linestyle="--", linewidth=1, label="moderado/alto")
ax.set_xlabel("Brecha CV train - validacion")
ax.set_title("Diagnostico de overfitting por modelo - cluster")
ax.legend()
fig.tight_layout()
fig.savefig(RUTA_GRAFICAS_CLUSTER / f"02_cluster_overfitting_gap_v2_binario_{DATASET_USAR}.png", dpi=220, bbox_inches="tight")
plt.show()


## Que revisar al terminar

Archivos principales:

- `modelo/V2/05_RESULTADOS/cluster/01_resultados_modelos_cluster_v2_binario_balanceado_organismo_clase.csv`
- `modelo/V2/05_RESULTADOS/cluster/05_cv_detalle_todos_modelos_cluster_v2_binario_balanceado_organismo_clase.csv`
- `modelo/V2/05_RESULTADOS/cluster/08_diagnostico_overfitting_cv_cluster_v2_binario_balanceado_organismo_clase.csv`
- `modelo/V2/GRAFICAS/cluster/01_cluster_ranking_modelos_v2_binario_balanceado_organismo_clase.png`
- `modelo/V2/GRAFICAS/cluster/02_cluster_overfitting_gap_v2_binario_balanceado_organismo_clase.png`

Criterio recomendado:

1. Mira `test_f1_macro`, `test_balanced_accuracy`, sensibilidad/especificidad de `Resistant` y `test_mse`.
2. Revisa ROC-AUC, PR-AUC, MCC y kappa cuando esten disponibles.
3. Revisa `gap_train_test`: si es mayor a `0.07`, hay alerta de overfitting.
4. Si dos modelos empatan, prefiere el de menor brecha CV y mejor sensibilidad de `Resistant`.

Estimacion para cluster de 60 hilos y 100GB RAM:

- Si todo corre bien: 4-12 horas.
- Si KNN/MLP/GradientBoosting/AdaBoost tardan mucho: 12-24 horas.
- Si necesitas recortar tiempo, empieza corriendo solo: `random_forest`, `extra_trees`, `hist_gradient_boosting`, `xgboost`.
